# 04 · Modeling — classification end to end

**Dataset:** `data/titanic_clean.csv`.
**Covers:** baselines · model choice · cross-validation · metrics · tuning ·
over/underfitting · feature importance.
**Time yourself:** ~40 minutes.

The setup cell below does the feature work from notebooks 01–03 for you, so you can
spend the time on modeling.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

df = pd.read_csv('data/titanic_clean.csv')
df.columns = df.columns.str.lower()

# --- features (from notebooks 01-03) ---
df['family_size'] = df['sibsp'] + df['parch'] + 1
df['is_alone'] = (df['family_size'] == 1).astype(int)
df['family_bin'] = pd.cut(df['family_size'], bins=[0, 1, 4, 20],
                          labels=['alone', 'small', 'large'])
df['title'] = df['name'].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()
common = df['title'].value_counts()[lambda s: s >= 10].index
df['title'] = df['title'].where(df['title'].isin(common), 'Rare')
df['fare_per_person'] = df['fare'] / df['ticket'].map(df['ticket'].value_counts())
df['has_cabin'] = df['cabin'].notna().astype(int)

NUMERIC = ['pclass', 'age', 'fare_per_person', 'is_alone', 'has_cabin']
CATEGORICAL = ['sex', 'embarked', 'title', 'family_bin']
FEATURES = NUMERIC + CATEGORICAL

X = df[FEATURES]
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

def make_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler())]), NUMERIC),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                          ('encode', OneHotEncoder(handle_unknown='ignore',
                                                   drop='first'))]), CATEGORICAL),
    ])

print(X_train.shape, X_test.shape, '| survival rate:', round(y.mean(), 3))
X_train.head()

---

## Part A — Baselines

### Q1. Before any model: what accuracy does a model that always predicts the majority class
get? This is the number every result must beat.

<details><summary>hint</summary>

`DummyClassifier(strategy='most_frequent')`. Quoting this number unprompted is a strong signal in an interview.

</details>

In [ ]:
# your code here

### Q2. Fit a logistic regression in a pipeline. Report train and test accuracy.
Is it beating the dummy?

<details><summary>hint</summary>

Wrap `make_preprocessor()` and the classifier in a `Pipeline`. Use `max_iter=1000` — the default 100 won't converge and will warn at you.

</details>

In [ ]:
# your code here

---

## Part B — Cross-validation

### Q3. A single split on 891 rows is noisy. Cross-validate the logistic regression with
5-fold stratified CV. Report mean **and** std. Why `StratifiedKFold` and not `KFold`?

<details><summary>hint</summary>

`cross_val_score(pipe, X_train, y_train, cv=cv)`. The std is what tells you whether a difference between two models is real.

</details>

In [ ]:
# your code here

### Q4. Compare 5 model families with the same CV, in one loop: logistic regression, decision
tree, random forest, gradient boosting, and KNN. Report mean ± std for each, sorted.
Which wins — and is the win larger than the noise?

<details><summary>hint</summary>

Build the dict, loop, collect into a DataFrame. Then compare the gaps against the stds before you announce a winner.

</details>

In [ ]:
# your code here

---

## Part C — Metrics

### Q5. Accuracy hides things. Print a full classification report and a confusion matrix for
the logistic regression. Which class does it fail on, and in which direction?

<details><summary>hint</summary>

`classification_report` gives precision/recall/F1 per class. `ConfusionMatrixDisplay.from_estimator` plots it in one line.

</details>

In [ ]:
# your code here

### Q6. Compute ROC-AUC and average precision (PR-AUC). Plot the ROC curve.
Why does AUC use `predict_proba` rather than `predict`?

<details><summary>hint</summary>

`predict_proba(X)[:, 1]` is the positive-class probability. AUC is threshold-independent — that's the whole point of it.

</details>

In [ ]:
# your code here

### Q7. The default threshold is 0.5, which nothing about your problem chose for you.
Suppose missing a survivor is 3× worse than a false alarm. Find the threshold that
maximises recall subject to precision staying above 0.6.

<details><summary>hint</summary>

`precision_recall_curve` returns arrays of precision/recall per threshold. Note `thr` is one shorter than the other two — that's why the `np.r_[thr, 1.0]`.

</details>

In [ ]:
# your code here

### Q8. Conceptual, no code needed. Titanic is ~38% positive — mildly imbalanced. If instead
it were **0.5%** positive (say, fraud), which of accuracy / ROC-AUC / PR-AUC would
you report, and why?

<details><summary>hint</summary>

Write out the FPR formula and ask what happens to it when TN is 99.5% of the data. That's the whole argument.

</details>

In [ ]:
# your code here

---

## Part D — Tuning

### Q9. Tune a `RandomForestClassifier` with `GridSearchCV` over `n_estimators`, `max_depth`
and `min_samples_leaf`. Report the best params and the best CV score.
Note the `clf__` prefix — why is it needed?

<details><summary>hint</summary>

Params for a pipeline step are named `<step_name>__<param>`. `n_jobs=-1` uses all cores.

</details>

In [ ]:
# your code here

### Q10. That grid was 24 combos × 5 folds = 120 fits. Show you know the alternative: run
`RandomizedSearchCV` over a *wider* space with a fixed budget of 20 candidates.
When would you prefer it?

<details><summary>hint</summary>

`randint(low, high)` from scipy.stats gives a sampling distribution instead of a fixed list. `n_iter` is your compute budget.

</details>

In [ ]:
# your code here

---

## Part E — Diagnosis

### Q11. Deliberately overfit: fit an unconstrained decision tree and report train vs test.
Then fix it. Name the symptom and the cure.

<details><summary>hint</summary>

A decision tree with no `max_depth` will drive training accuracy toward 1.0. The gap between train and test is the diagnosis.

</details>

In [ ]:
# your code here

### Q12. Plot a learning curve for the tuned random forest. Does the model need **more data**
or **more capacity**? Justify from the shape of the curves.

<details><summary>hint</summary>

`learning_curve` returns `(sizes, train_scores, val_scores)` where the score arrays are (n_sizes, n_folds) — average across axis=1. Look at whether the validation curve is still climbing at the right-hand edge.

</details>

In [ ]:
# your code here

---

## Part F — Interpretation

### Q13. Which features matter? Get the random forest's importances. Then get **permutation
importance** on the test set. Where do they disagree, and which do you trust?

<details><summary>hint</summary>

`best_rf.named_steps['clf'].feature_importances_` for Gini; `permutation_importance(model, X_test, y_test, n_repeats=20)` for the other. `get_feature_names_out()` on the preprocessor recovers the one-hot names.

</details>

In [ ]:
# your code here

### Q14. Read the logistic regression's coefficients as odds ratios. Which feature has the
largest effect and what does its number literally mean?

The answer may not be the feature you expect — if so, explain why.

<details><summary>hint</summary>

`np.exp(coef)` converts a log-odds coefficient into an odds ratio. Before you interpret the winner, ask which *other* feature it's correlated with. Also remember the numeric features were standardised.

</details>

In [ ]:
# your code here

---

## Part G — Ship it

### Q15. Final checklist. Pick your final model, evaluate it **once** on the test set, compare
against the baseline, and save the whole pipeline to disk. Then state what you'd
monitor in production.

<details><summary>hint</summary>

`joblib.dump(pipeline, 'model.joblib')`. Save the *pipeline*, never the bare estimator.

</details>

In [ ]:
# your code here

---

## Debrief — the sentences worth having ready

- *"Baseline first."* Dummy 0.615 → logreg 0.827. That +21 points is the real result; the
  ensembles add nothing on top of it here.
- *"Is that gap bigger than the std?"* Almost always the right question when two CV
  numbers are compared. Here the top three models sit inside one std of each other — that
  is a tie, not a leaderboard.
- *"CV picked the RF; the test set disagreed."* Choosing the best of 24 candidates on CV
  bakes luck into the winner's score. It's why the test set stays sealed until the end.
- *"Accuracy is the wrong metric here because…"* — have the FPR argument ready.
- *"The threshold is a business decision."* Free performance, no retraining.
- *"Simplest model that hits the bar."* Being willing to *not* ship the fancy model reads
  as seniority.